## Data preparation for the RAG tool that will serve information to the model

Developer: Eliel Paes    
Last update: 2025-12-10

### Install

In [0]:
%pip install -r requirements.txt

dbutils.library.restartPython()

### 1 Imports

In [0]:
import io
import os
import pandas as pd
import json
import fitz

from typing import Iterator
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

### 1 Data preparation

#### 1.1 Extracting pdf raw text

In [0]:
# Defining some variables
articles_path = "/Volumes/workspace/default/study_files"
catalog = "workspace"
db_name = "default"
table_name = f"pdf_raw_text"

# reading pdf files as binary
# df schema will be: 
      # |-- path: string (nullable = true)
      # |-- modificationTime: timestamp (nullable = true)
      # |-- length: long (nullable = true)
      # |-- content: binary (nullable = true)      
df = (spark.read.format("binaryFile")
      .option("recursiveFileLookup", "true")
      .load(articles_path))

#### Functions to be apllied in the content column in order to extract the pad raw text

In [0]:
@pandas_udf("array<string>")
def get_pdf_raw_text(contents: Iterator[pd.Series]) -> Iterator[pd.Series]:
    """
        Text extraction from pdf files. 

        The content of pdf files are in the content column of our pyspark dataframe, so we will iterate over the content column and extract the text from each pdf file.    
    """     
    def extract_doc_text(binary_col):
        """
            receives e pdf (bytes) and return the extracted text.
        
        """    
        content = []
        try:
            pdf = fitz.open(stream = binary_col)
            for page in pdf.pages():
                content.append(str(json.dumps({"page": page.number + 1, "content": page.get_text()})))

            return content       
            
        except Exception as e:
            print(f"Erro ao carregar arquivo {binary_col}: {e}")

    for batch in contents: 
        yield batch.apply(extract_doc_text)    


@pandas_udf("int")
def get_page_number(pdf_pages: Iterator[pd.Series]) -> Iterator[pd.Series]:
    """
    
    """        
    def get_topics(col):
        """
        
        
        """
        print(col)
        text = json.loads(col)
        return text["page"]
    
    for page in pdf_pages:
        yield page.apply(get_topics)        
        

In [0]:
# get pdf path and binary content
df_raw_text = df.select("path", "content")

# Extract the pdf raw text for each page and explode the array
df_raw_text = df_raw_text.withColumn("pdf_text_page_content", get_pdf_raw_text(F.col("content"))).drop("content")
df_raw_text = df_raw_text.withColumn("pdf_text_page_content", F.explode(F.col("pdf_text_page_content")))

# Get the page number
df_raw_text = df_raw_text.withColumn("page", get_page_number(F.col("pdf_text_page_content"))) 

df_raw_text.select("path", "page", "pdf_text_page_content").write.mode("overwrite").saveAsTable(f"{catalog}.{db_name}.{table_name}")